# Assignment 1. Exploratory Modeling with JUSTICE

**Course:** EPA141A Model Based Decision Making — Delft University of Technology  
**Model:** JUSTICE (development)


## Learning objectives

After completing this assignment you will be able to:
1. Verify that the JUSTICE model is installed and runs correctly.
2. Identify and define **model input uncertainties** — including physical and normative parameters.
3. Use **Latin Hypercube Sampling** to generate a set of scenarios that spans the uncertainty space.
4. Run JUSTICE across many scenarios and collect the outcomes.
5. Visualise outcome **distributions** and describe how parameter uncertainty propagates into model outputs.


## Background

When building models of complex systems (climate, economy, ecology) we rarely know
the exact value of every parameter. A damage-function coefficient, a discount rate,
or a climate sensitivity parameter are all uncertain. **Exploratory modeling** treats
uncertainty explicitly: we run *many* scenarios, sampling the uncertain parameters
systematically, and study the resulting spread in model outputs.

In this assignment we treat four parameters as uncertain:

| Symbol | Parameter | Default | Range | Description |
|--------|-----------|---------|-------|-------------|
| `ρ` | Pure rate of social time preference | 0.015 | [0.001, 0.030] | Contested in the Stern–Nordhaus debate |
| `η` | Elasticity of marginal utility | 1.45 | [0.5, 1.5] | Empirical estimates range widely |
| `δ` | Damage function scale factor | 1.0 | [0.5, 2.0] | Structural uncertainty in damage estimates |
| `ecs_ensemble` | FaIR ensemble member index | 1 | [1, 1001] | Selects a calibrated ECS parameter set |

`ecs_ensemble` is a *physical* uncertainty (how sensitive the climate system is to CO₂),
while `ρ`, `η`, and `δ` are *normative* uncertainties (contested values in economic ethics).

### Workflow note
This version uses the **EMA Workbench** (`perform_experiments`) to handle LHS sampling
and model execution automatically. The conceptual goals — defining uncertainties, generating
a scenario ensemble, and visualising outcome distributions — are identical to a manual LHS workflow.

## Step 0 — Verify your installation

Run the cell below. Every line should end with ✓.  
If any line shows ✗, follow the installation instructions in `JUSTICE_analysis_workflow.ipynb`.

In [1]:
import importlib, sys
required = ["justice", "numpy", "pandas", "matplotlib",
            "ema_workbench", "scipy", "seaborn"]
for pkg in required:
    found = importlib.util.find_spec(pkg) is not None
    print(f"  {'✓' if found else '✗'}  {pkg}")
print(f"\nPython {sys.version.split()[0]}")

  ✗  justice
  ✓  numpy
  ✓  pandas
  ✓  matplotlib
  ✓  ema_workbench
  ✓  scipy
  ✓  seaborn

Python 3.12.13


## Step 1 — Run JUSTICE with default parameters

Before exploring uncertainty, confirm the model runs with its default settings.

**Task 1.1** — Complete the `justice_model` function below. It should:
1. Hard-reset JUSTICE and instantiate a fresh model with the given `ecs_ensemble` index.
2. Set `rho` on `model.welfare_function.pure_rate_of_social_time_preference`.
3. Set `eta` on `model.welfare_function.elasticity_of_marginal_utility_of_consumption`.
4. Scale the three active damage coefficients (`coefficient_a`, `coefficient_b`, `damage_gdp_ratio_with_gradient`) by `delta`.
5. Run with zero abatement (`ECR = 0`) and return all four outcomes as a dict.

> **Note:** Default argument values are required by the EMA Workbench function model interface.

**Task 1.2** — Run the default case (`ecs_ensemble=1`, all other parameters at their defaults) and record the four outcome values in the table below.

| Outcome | Default value |
|---------|--------------|
| `welfare` | |
| `years_above_temperature_threshold` | |
| `welfare_loss_damage` | |
| `welfare_loss_abatement` | |

In [3]:
import os, sys, logging
import warnings; warnings.filterwarnings("ignore")

# ── Resolve notebook location and add JUSTICE-main to sys.path ─────────────
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_justice_root = os.path.normpath(os.path.join(_NOTEBOOK_DIR, '../JUSTICE-main'))
_PLOTS_DIR = os.path.join(_NOTEBOOK_DIR, "plots")
os.makedirs(_PLOTS_DIR, exist_ok=True)
if _justice_root not in sys.path:
    sys.path.insert(0, _justice_root)
os.chdir(_justice_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from ema_workbench import (
    Model, RealParameter, ScalarOutcome, ArrayOutcome,
    perform_experiments, ema_logging, Sample,
    SequentialEvaluator, MultiprocessingEvaluator,
)
from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction
from justice.objectives.objective_functions import years_above_temperature_threshold

ema_logging.log_to_stderr(logging.WARNING)

# ── Python 3.14 compatibility patch for matplotlib.path.Path.__deepcopy__ ──
import matplotlib.path as _mpath
def _patched_path_deepcopy(self, memo=None):
    if memo is None: memo = {}
    new_path = _mpath.Path.__new__(_mpath.Path)
    memo[id(self)] = new_path
    verts = self._vertices.copy()
    codes = self._codes.copy() if self._codes is not None else None
    new_path.__init__(verts, codes,
                      _interpolation_steps=self._interpolation_steps, readonly=False)
    return new_path
_mpath.Path.__deepcopy__ = _patched_path_deepcopy

print("Imports OK")

Imports OK


In [4]:
def justice_model(rho=0.015, eta=1.45, delta=1.0, ecs_ensemble=1, ecr_plateau=0.0):
    """
    EMA Workbench function model interface for JUSTICE.

    Parameters
    ----------
    rho           : float — pure rate of social time preference
    eta           : float — elasticity of marginal utility of consumption
    delta         : float — output damage exponent
    ecs_ensemble  : float — FaIR ensemble index (1–1001); cast to int inside
    ecr_plateau   : float — uniform emission control rate (0=BAU, 1=full abatement)

    Returns
    -------
    dict with keys: welfare, years_above_temperature_threshold,
                    welfare_loss_damage, welfare_loss_abatement,
                    temperature_trajectory
    """
    JUSTICE.hard_reset()
    ensemble_idx = int(np.round(np.clip(ecs_ensemble, 1, 1001)))

    model = JUSTICE(
        start_year=2015, end_year=2300, timestep=1,
        scenario=2,
        climate_ensembles=ensemble_idx,
        stochastic_run=False,
        social_welfare_function=WelfareFunction.UTILITARIAN,
    )

    # Set a uniform ECR across all regions and timesteps, then run the model
    ecr = np.full((model.no_of_regions, model.no_of_timesteps), ecr_plateau)
    model.run(ecr)
    data = model.evaluate()
    return {
        "welfare":                           data["welfare"][0],
        "years_above_temperature_threshold": data["years_above_temperature_threshold"][0],
        "welfare_loss_damage":               data["welfare_loss_damage"][0],
        "welfare_loss_abatement":            data["welfare_loss_abatement"][0],
        "temperature_trajectory":            data["global_temperature"][:, 0],
    }

In [ ]:
justice_model()

## Step 2 — Define uncertain parameters

**Task 2.1** — Complete the cell below by:
1. Creating a `Model` object wrapping `justice_model`.
2. Assigning `uncertainties` — four `RealParameter` objects matching the table in the Background.
3. Assigning `outcomes` — four `ScalarOutcome` objects.

**Task 2.2** — Explain in 2–3 sentences why a *higher* pure rate of time preference `ρ` tends to lead to *less* stringent near-term mitigation. Reference the Ramsey rule in your answer.

*(Write your answer here)*

**Task 2.3** — Explain why `ecs_ensemble` is a *physical* uncertainty while `ρ`, `η`, `δ` are *normative* uncertainties. What does this distinction imply about which outcomes each type of uncertainty can influence?

*(Write your answer here)*

In [ ]:
em_model = Model('JUSTICE', function=justice_model)

# Define four uncertain parameters.
# Choose bounds that reflect the plausible range of each parameter.
# Hint: rho is a discount rate, eta is an elasticity, delta scales damages,
#       ecs_ensemble indexes the FaIR climate sensitivity ensemble (1–1001).
em_model.uncertainties = [
    RealParameter('rho',          ..., ...),   # pure rate of social time preference
    RealParameter('eta',          ..., ...),   # elasticity of marginal utility
    RealParameter('delta',        ..., ...),   # output damage exponent
    RealParameter('ecs_ensemble', ..., ...),   # FaIR ensemble index
]

em_model.levers = [
    RealParameter('ecr_plateau', 0.0, 1.0),
]

em_model.outcomes = [
    ScalarOutcome('welfare'),
    ScalarOutcome('years_above_temperature_threshold'),
    ScalarOutcome('welfare_loss_damage'),
    ScalarOutcome('welfare_loss_abatement'),
    ArrayOutcome('temperature_trajectory'),
]

print(em_model)

## Step 3 — Latin Hypercube Sampling ensemble

**Latin Hypercube Sampling (LHS)** divides each uncertain dimension into equally
probable intervals and samples exactly once from each interval. Compared to Monte Carlo,
it guarantees better coverage of the full parameter space with fewer samples.

`perform_experiments` uses LHS by default. A single call handles sampling, execution, and result collection.

**Task 3.1** — Run 100 scenarios using `SequentialEvaluator`. Store the results as:
- `experiments` — a DataFrame with one row per scenario and one column per parameter
- `outcomes` — a dict mapping outcome names to arrays

In [ ]:
# Choose two policies to compare (e.g. BAU vs. a moderate abatement level)
# and choose how many LHS scenarios to sample.
policies = [
    Sample('no_abatement',       ecr_plateau=...),   # no abatement
    Sample('moderate_abatement', ecr_plateau=...),   # choose a plateau level
]

with SequentialEvaluator(em_model) as evaluator:
    experiments, outcomes = evaluator.perform_experiments(
        scenarios=..., policies=policies)   # choose number of scenarios

df_results = pd.DataFrame(outcomes)
df_results['policy'] = experiments['policy'].values

print(f"Experiments: {experiments.shape}")
print(f"Outcomes:    {list(outcomes.keys())}")
df_results.head()

## Step 4 — Visualise outcome distributions

**Task 4.1** — Plot a 2×2 grid of histograms showing the distribution of each
outcome across the 100 scenarios. Mark the **default run value** from Step 1
as a vertical dashed line on each panel.

**Task 4.2** — Plot a scatter matrix (pairplot) of the four uncertain parameters
coloured by `welfare` quartile. Use the `experiments` DataFrame to extract parameter columns.

*Hint: use `seaborn.pairplot` with `hue`.*

In [ ]:
OBJECTIVES = ["welfare", "years_above_temperature_threshold",
              "welfare_loss_damage", "welfare_loss_abatement"]
PARAMS = ["rho", "eta", "delta", "ecs_ensemble"]

defaults = justice_model()
palette = {'no_abatement': 'steelblue', 'moderate_abatement': 'darkorange'}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, obj in zip(axes.flat, OBJECTIVES):
    for pol, grp in df_results.groupby('policy'):
        # Hint: ax.hist(grp[obj], bins=25, ...)
        ...
    ax.axvline(defaults[obj], color='black', lw=1.5, ls='--', label='default')
    ax.set_xlabel(obj)
    ax.set_ylabel('count')
    ax.legend(fontsize=8)

fig.suptitle('Outcome distributions — BAU vs. moderate abatement', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", "a01_histograms.png"), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
df_plot = experiments[PARAMS].copy()
for obj in OBJECTIVES:
    df_plot[obj] = outcomes[obj]
df_plot['welfare_q'] = pd.qcut(df_plot['welfare'], q=4, labels=['Q1','Q2','Q3','Q4'])

# Hint: sns.pairplot(df_plot, vars=PARAMS, hue='welfare_q', ...)
...

plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", "a01_pairplot.png"), dpi=120, bbox_inches='tight')
plt.show()

## Reflection Questions

**1. Coverage vs. efficiency.** Why does LHS typically need fewer samples than simple random (Monte Carlo) sampling to achieve the same coverage of the parameter space?


*(Write your answer here.)*


**2. Default as reference.** In your histogram plots, does the default run fall near the centre of the distribution or near one of the tails? What does this imply about how representative a single default run is?


*(Write your answer here.)*


**3. Propagation of uncertainty.** Which of the four outcomes shows the widest relative spread? Propose a hypothesis explaining why that outcome is more sensitive to the chosen uncertain parameters.


*(Write your answer here.)*